# Checkpoint D1: Chay danh gia day du
Ket qua chay evaluator tren 12 cau hoi voi simulated RAG pipeline.

In [ ]:
import csv
import json
import re
from pathlib import Path
from collections import Counter

print('Imports OK: csv, json, re, pathlib, collections')

In [ ]:
# Load test questions
questions_path = Path('../../synthetic-data/test-questions.csv')
questions = []
with open(questions_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        questions.append(row)

print(f'Loaded {len(questions)} test questions')
cat_counts = Counter(q.get('category', '') for q in questions)
print(f'Categories: {dict(cat_counts)}')

In [ ]:
# Load 4 HR policies
policy_dir = Path('../../synthetic-data/hr-policies')
policies = {}
for f in sorted(policy_dir.glob('*.md')):
    content = f.read_text(encoding='utf-8')
    doc_id_match = re.search(r'doc_id:\s*(\S+)', content)
    doc_id = doc_id_match.group(1) if doc_id_match else f.stem
    policies[doc_id] = {
        'filename': f.name,
        'content': content,
        'doc_id': doc_id
    }
    print(f'Loaded: {f.name} → {doc_id}')

print(f'\nTotal: {len(policies)} policies loaded')

# Create chunks (heading-based)
chunks = []
for doc_id, pol in policies.items():
    content = pol['content']
    sections = re.split(r'(^## .+$)', content, flags=re.MULTILINE)
    
    for i in range(0, len(sections) - 1, 2):
        heading = sections[i].strip() if i > 0 else 'Header'
        body = sections[i + 1].strip() if i + 1 < len(sections) else ''
        heading_clean = re.sub(r'^#+\s*', '', heading)
        chunks.append({
            'doc_id': doc_id,
            'filename': pol['filename'],
            'heading': heading_clean,
            'text': (heading + '\n' + body).strip()
        })

print(f'Total chunks: {len(chunks)}')

In [ ]:
# Simulated RAG pipeline (mock)

# Pre-defined correct answers based on the actual policy content
ANSWER_DB = {
    'Q-001': {
        'answer': 'Nhan vien chinh thuc duoc huong ngay phep nam theo tham nien: Duoi 5 nam: 12 ngay, Tu 5 den duoi 10 nam: 14 ngay, Tu 10 nam tro len: 16 ngay. Nhan vien thu viec khong duoc huong ngay phep nam.',
        'citation': 'POL-LEAVE-001 muc 1.1',
        'quote': 'Nhan vien chinh thuc duoc huong ngay phep nam theo tham nien: Duoi 5 nam: 12 ngay'
    },
    'Q-002': {
        'answer': 'Muc phu cap an trua la 30.000 VNĐ/ngay cho nhan vien chinh thuc va nhan vien thu viec. Thuc tap sinh duoc 20.000 VNĐ/ngay.',
        'citation': 'POL-ALLOW-001 muc 1.1',
        'quote': 'Nhan vien chinh thuc: 30.000 VNĐ'
    },
    'Q-003': {
        'answer': 'Quy trinh xin nghi phep gom 4 buoc: (1) Gui yeu cau qua he thong noi bo truoc it nhat 3 ngay lam viec, (2) Quan ly truc tiep phe duyet trong vong 1 ngay lam viec, (3) Nhan su xac nhan va ghi nhan vao he thong, (4) Nghi phep lien tuc toi da 5 ngay.',
        'citation': 'POL-LEAVE-001 muc 1.3',
        'quote': 'Gui yeu cau qua he thong noi bo truoc it nhat 3 ngay lam viec'
    },
    'Q-004': {
        'answer': 'Tham nien 6 nam (bac 3: 5-10 nam) duoc 14 ngay phep nam (co ban) + 2 ngay phep them (tham nien 5-10 nam) = tong cong 16 ngay phep nam.',
        'citation': 'POL-LEAVE-001 muc 1.1 + POL-SENIOR-001 muc 3.1',
        'quote': 'Tu 5 den duoi 10 nam: 14 ngay. Nhan vien o bac tham nien 3 tro len (5 nam+) duoc them ngay phep: 5-10 nam: +2 ngay'
    },
    'Q-005': {
        'answer': 'KHONG. Nhan vien thu viec khong duoc huong phu cap dien thoai. Chi ap dung cho nhan vien chinh thuc sau thoi gian thu viec.',
        'citation': 'POL-ALLOW-001 muc 3.2',
        'quote': 'Chi ap dung cho nhan vien chinh thuc sau thoi gian thu viec. Nhan vien thu viec khong duoc huong phu cap dien thoai.'
    },
    'Q-006': {
        'answer': 'CO. Cong ty xem xet tai tro mot phan hoc phi cho chuong trinh MBA hoac thac si. Muc tai tro toi da: 50% hoc phi, khong vuot qua 100.000.000 VNĐ. Dieu kien: da lam viec du 2 nam, co ket qua danh gia hieu suat dat loai kha tro len trong 2 nam gan nhat.',
        'citation': 'POL-TRAIN-001 muc 4.1',
        'quote': 'Cong ty xem xet tai tro mot phan hoc phi cho chuong trinh MBA. Muc tai tro toi da: 50% hoc phi'
    },
    'Q-007': {
        'answer': 'Nghi om qua 30 ngay thi tu ngay thu 31 tro di huong 70% luong, toi da 180 ngay/nam. Can giay chung nhan nghi om cua benh vien hoac phong kham co giay phep.',
        'citation': 'POL-LEAVE-001 muc 2.1',
        'quote': 'Tu ngay thu 31 tro di huong 70% luong, toi da 180 ngay/nam'
    },
    'Q-008': {
        'answer': 'CO. Khi dang nghi viec, phep du duoc quy doi thanh tien. Phep nam chua su dung co the chuyen sang nam sau (toi da 50%), nhung phep du khong duoc quy doi thanh tien tru truong hop nghi viec.',
        'citation': 'POL-LEAVE-001 muc 1.2',
        'quote': 'Phep du khong duoc quy doi thanh tien tru truong hop nghi viec'
    },
    'Q-009': {
        'answer': 'Cau hoi "Chinh sach cua cong ty co tot khong?" chua duoc cu the. Vui long lam ro ban muon hoi ve chinh sach nao (nghi phep, phu cap, dao tao, tham nien) de toi co the tra loi chinh xac.',
        'citation': None,
        'quote': None
    },
    'Q-010': {
        'answer': 'Cau hoi chua ro loai nghi. Vui long lam ro ban muon hoi ve nghi phep (huong luong), nghi om (huong luong 30 ngay dau), hay nghi khong luong. Moi loai nghi co che do luong khac nhau.',
        'citation': None,
        'quote': None
    },
    'Q-011': {
        'answer': 'Xin loi, toi khong co thong tin ve chinh sach bao hiem xa hoi trong co so du lieu. Vui long lien he Phong Nhan su de duoc ho tro.',
        'citation': None,
        'quote': None
    },
    'Q-012': {
        'answer': 'Xin loi, chinh sach chuyen cong tac khong nam trong pham vi co so du lieu cua toi. Vui long lien he Phong Nhan su hoac quan ly truc tiep de duoc huong dan.',
        'citation': None,
        'quote': None
    }
}

print(f'Simulated RAG pipeline ready with {len(ANSWER_DB)} pre-defined answers.')
print('Categories handled:')
print('  Q-001 to Q-008: in-scope → answer with citation')
print('  Q-009 to Q-010: ambiguous → clarification request')
print('  Q-011 to Q-012: out-of-scope → refusal + HR contact')

In [ ]:
# Run simulated pipeline on all 12 questions
pipeline_results = []

for q in questions:
    qid = q.get('question_id', '')
    answer_data = ANSWER_DB.get(qid, {})
    
    pipeline_results.append({
        'question_id': qid,
        'category': q.get('category', ''),
        'question': q.get('question', ''),
        'answer': answer_data.get('answer', 'No answer available'),
        'citation': answer_data.get('citation'),
        'quote': answer_data.get('quote'),
        'expected_source': q.get('expected_source', ''),
        'expected_behavior': q.get('expected_behavior', '')
    })

print(f'Ran simulated pipeline on {len(pipeline_results)} questions.')
print()
for r in pipeline_results[:3]:
    print(f'{r["question_id"]}: {r["question"][:50]}...')
    print(f'  Answer: {r["answer"][:80]}...')
    print(f'  Citation: {r["citation"]}')
    print()
print('... (12 questions total)')

In [ ]:
# Evaluate each answer

def cross_check_citation(quote, source_content):
    """Verify quote exists in source text."""
    if not quote or not source_content:
        return False
    phrases = re.findall(r'[\w\s]{4,}', quote)
    phrases = [p.strip() for p in phrases if len(p.strip()) > 3]
    if not phrases:
        return False
    matched = sum(1 for p in phrases[:5] if p.lower() in source_content.lower())
    return matched >= min(len(phrases), 5) * 0.5

eval_results = []

for r in pipeline_results:
    cat = r['category']
    checks = {}
    all_pass = True
    reasons = []
    
    if 'out-of-scope' in cat:
        # Check for refusal
        has_refusal = any(kw in r['answer'].lower() for kw in [
            'khong co', 'khong nam', 'ngoai pham vi', 'lien he nhan su', 'xin loi'
        ])
        checks['refusal'] = has_refusal
        if not has_refusal:
            all_pass = False
            reasons.append('No refusal')
    
    elif 'ambiguous' in cat:
        # Check for clarification
        has_clarification = any(kw in r['answer'].lower() for kw in [
            'lam ro', 'cu the', 'xac dinh', 'vui long', 'khong ro'
        ])
        checks['clarification'] = has_clarification
        if not has_clarification:
            all_pass = False
            reasons.append('No clarification')
    
    else:
        # In-scope: check content, citation, cross-check
        checks['has_content'] = len(r['answer']) > 50
        checks['has_citation'] = r['citation'] is not None
        
        # Cross-check quote against source policy
        quote = r.get('quote')
        source_doc = r.get('expected_source', '')
        # Find matching policy
        source_text = ''
        for doc_id, pol in policies.items():
            if doc_id in source_doc:
                source_text = pol['content']
                break
        checks['quote_verified'] = cross_check_citation(quote, source_text) if quote else False
        
        if not checks['has_content']:
            all_pass = False
            reasons.append('No content')
        if not checks['has_citation']:
            all_pass = False
            reasons.append('No citation')
        if not checks['quote_verified']:
            all_pass = False
            reasons.append('Quote not verified')
    
    eval_results.append({
        'question_id': r['question_id'],
        'category': cat,
        'question': r['question'],
        'checks': checks,
        'pass': all_pass,
        'reasons': reasons
    })

print('Evaluation complete.')

In [ ]:
# Generate SLI metrics

in_scope = [r for r in eval_results if 'in-scope' in r['category']]
out_scope = [r for r in eval_results if 'out-of-scope' in r['category']]
ambiguous = [r for r in eval_results if 'ambiguous' in r['category']]

# Calculate metrics
sli = {
    'in_scope_accuracy': sum(1 for r in in_scope if r['pass']) / len(in_scope) if in_scope else 0,
    'out_scope_refusal': sum(1 for r in out_scope if r['pass']) / len(out_scope) if out_scope else 0,
    'citation_rate': sum(1 for r in in_scope if r['checks'].get('has_citation')) / len(in_scope) if in_scope else 0,
    'quote_verification': sum(1 for r in in_scope if r['checks'].get('quote_verified')) / len(in_scope) if in_scope else 0,
    'ambiguous_handling': sum(1 for r in ambiguous if r['pass']) / len(ambiguous) if ambiguous else 0,
}

# SLO targets
slo = {
    'in_scope_accuracy': 0.90,
    'out_scope_refusal': 0.95,
    'citation_rate': 0.90,
    'quote_verification': 0.80,
    'ambiguous_handling': 0.80,
}

print('=' * 70)
print(f'{"SLI Metric":<30} {"SLI Value":<12} {"SLO Target":<12} {"Status"}')
print('=' * 70)

for metric in sli:
    sli_val = sli[metric]
    slo_val = slo[metric]
    status = 'PASS' if sli_val >= slo_val else 'FAIL'
    print(f'{metric:<30} {sli_val:>6.0%}      {slo_val:>6.0%}      {status}')

all_sli_pass = all(sli[m] >= slo[m] for m in sli)
print('=' * 70)
print(f'Overall SLO: {"PASS" if all_sli_pass else "FAIL"}')

In [ ]:
# Formatted evaluation report
print('=' * 70)
print('FULL EVALUATION REPORT')
print('=' * 70)
print()
print(f'{"ID":<8} {"Category":<20} {"Result":<8} {"Checks":<35} {"Issues"}')
print('-' * 90)

for r in eval_results:
    status = 'PASS' if r['pass'] else 'FAIL'
    checks_str = ', '.join(f'{k}={"Y" if v else "N"}' for k, v in r['checks'].items())
    issues = '; '.join(r['reasons']) if r['reasons'] else '-'
    print(f'{r["question_id"]:<8} {r["category"]:<20} {status:<8} {checks_str:<35} {issues}')

print('-' * 90)
pass_count = sum(1 for r in eval_results if r['pass'])
print(f'Summary: {pass_count}/{len(eval_results)} questions PASSED ({pass_count/len(eval_results):.0%})')
print()
print('SLI vs SLO:')
for metric in sli:
    sli_val = sli[metric]
    slo_val = slo[metric]
    indicator = 'OK' if sli_val >= slo_val else 'BELOW TARGET'
    print(f'  {metric}: {sli_val:.0%} (target: {slo_val:.0%}) [{indicator}]')

In [ ]:
print('=' * 60)
print('CHECKPOINT D1 SUMMARY')
print('=' * 60)
print()
print(f'Questions evaluated: {len(eval_results)}')
print(f'Passed: {pass_count}/{len(eval_results)} ({pass_count/len(eval_results):.0%})')
print(f'Overall SLO: {"PASS" if all_sli_pass else "FAIL"}')
print()
print('Day la ket qua mo phong voi pre-defined answers.')
print('De chay thuc te, can:')
print('  1. Gemini API — de generate cau tra loi thuc')
print('  2. ChromaDB + sentence-transformers — cho vector search')
print('  3. Chay scripts/evaluator.py trong Skill package')
print()
print('NEXT: Chay checkpoint-step-d2.ipynb de chuan bi cross-team testing.')